In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# set paths

project_dir = r"C:\Users\Kimi\OneDrive\Desktop\PORTFOLIO PROJECTS\Health_Insurance_claims_member_engagement"

processed_dir = os.path.join(project_dir, "data_processed")
outputs_dir = os.path.join(project_dir, "outputs")
tables_dir = os.path.join(outputs_dir, "tables")

os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

print("Processed dir:", processed_dir)

Processed dir: C:\Users\Kimi\OneDrive\Desktop\PORTFOLIO PROJECTS\Health_Insurance_claims_member_engagement\data_processed


In [3]:
# load cleaned files

benef_2009 = pd.read_csv(os.path.join(processed_dir, "benef_2009_small.csv"), low_memory=False)
benef_2010 = pd.read_csv(os.path.join(processed_dir, "benef_2010_small.csv"), low_memory=False)
inp = pd.read_csv(os.path.join(processed_dir, "inpatient_small.csv"), low_memory=False)
outp = pd.read_csv(os.path.join(processed_dir, "outpatient_small.csv"), low_memory=False)

print("benef_2009:", benef_2009.shape)
print("benef_2010:", benef_2010.shape)
print("inp:", inp.shape)
print("outp:", outp.shape)

benef_2009: (114538, 32)
benef_2010: (112754, 32)
inp: (66773, 35)
outp: (790790, 31)


In [4]:
# re-convert date columns after reload

benef_date_cols = ["BENE_BIRTH_DT", "BENE_DEATH_DT"]
inp_date_cols = ["CLM_FROM_DT", "CLM_THRU_DT", "CLM_ADMSN_DT", "NCH_BENE_DSCHRG_DT"]
outp_date_cols = ["CLM_FROM_DT", "CLM_THRU_DT"]

for col in benef_date_cols:
    benef_2009[col] = pd.to_datetime(benef_2009[col], errors="coerce")
    benef_2010[col] = pd.to_datetime(benef_2010[col], errors="coerce")

for col in inp_date_cols:
    inp[col] = pd.to_datetime(inp[col], errors="coerce")

for col in outp_date_cols:
    outp[col] = pd.to_datetime(outp[col], errors="coerce")

In [5]:
# sanity check

print(inp[["DESYNPUF_ID", "CLM_ID", "CLM_FROM_DT", "CLM_PMT_AMT"]].head())
print(outp[["DESYNPUF_ID", "CLM_ID", "CLM_FROM_DT", "CLM_PMT_AMT"]].head())

        DESYNPUF_ID           CLM_ID                   CLM_FROM_DT  \
0  00013D2EFD8E45D1  196661176988405 1970-01-01 00:00:00.020100312   
1  00016F745862898F  196201177000368 1970-01-01 00:00:00.020090412   
2  00016F745862898F  196661177015632 1970-01-01 00:00:00.020090831   
3  00016F745862898F  196091176981058 1970-01-01 00:00:00.020090917   
4  00016F745862898F  196261176983265 1970-01-01 00:00:00.020100626   

   CLM_PMT_AMT  
0       4000.0  
1      26000.0  
2       5000.0  
3       5000.0  
4      16000.0  
        DESYNPUF_ID           CLM_ID                   CLM_FROM_DT  \
0  00013D2EFD8E45D1  542192281063886 1970-01-01 00:00:00.020080904   
1  00016F745862898F  542272281166593 1970-01-01 00:00:00.020090602   
2  00016F745862898F  542282281644416 1970-01-01 00:00:00.020090623   
3  0001FDD721E223DC  542642281250669 1970-01-01 00:00:00.020091011   
4  00024B3D2352D2D0  542242281386963 1970-01-01 00:00:00.020080712   

   CLM_PMT_AMT  
0         50.0  
1         30.0  
2    

In [6]:
# create inpatient diagnosis count

inp_diag_cols = [
    "ADMTNG_ICD9_DGNS_CD",
    "ICD9_DGNS_CD_1", "ICD9_DGNS_CD_2", "ICD9_DGNS_CD_3", "ICD9_DGNS_CD_4",
    "ICD9_DGNS_CD_5", "ICD9_DGNS_CD_6", "ICD9_DGNS_CD_7", "ICD9_DGNS_CD_8",
    "ICD9_DGNS_CD_9", "ICD9_DGNS_CD_10"
]

inp["inpatient_diag_count"] = inp[inp_diag_cols].notna().sum(axis=1)

In [7]:
# create outpatient diagnosis count

outp_diag_cols = [
    "ADMTNG_ICD9_DGNS_CD",
    "ICD9_DGNS_CD_1", "ICD9_DGNS_CD_2", "ICD9_DGNS_CD_3", "ICD9_DGNS_CD_4",
    "ICD9_DGNS_CD_5", "ICD9_DGNS_CD_6", "ICD9_DGNS_CD_7", "ICD9_DGNS_CD_8",
    "ICD9_DGNS_CD_9", "ICD9_DGNS_CD_10"
]

outp["outpatient_diag_count"] = outp[outp_diag_cols].notna().sum(axis=1)

In [8]:
# create inpatient member-level features

inp_features = (
    inp.groupby("DESYNPUF_ID")
    .agg(
        inpatient_claim_count=("CLM_ID", "count"),
        inpatient_total_paid=("CLM_PMT_AMT", "sum"),
        inpatient_avg_paid=("CLM_PMT_AMT", "mean"),
        inpatient_max_paid=("CLM_PMT_AMT", "max"),
        inpatient_total_primary_payer_paid=("NCH_PRMRY_PYR_CLM_PD_AMT", "sum"),
        inpatient_total_deductible=("NCH_BENE_IP_DDCTBL_AMT", "sum"),
        inpatient_total_coinsurance=("NCH_BENE_PTA_COINSRNC_LBLTY_AM", "sum"),
        inpatient_total_blood_deductible=("NCH_BENE_BLOOD_DDCTBL_LBLTY_AM", "sum"),
        inpatient_total_util_days=("CLM_UTLZTN_DAY_CNT", "sum"),
        inpatient_avg_diag_count=("inpatient_diag_count", "mean")
    )
    .reset_index()
)

inp_features["has_inpatient_claim"] = 1

inp_features.head()

,DESYNPUF_ID,inpatient_claim_count,inpatient_total_paid,inpatient_avg_paid,inpatient_max_paid,inpatient_total_primary_payer_paid,inpatient_total_deductible,inpatient_total_coinsurance,inpatient_total_blood_deductible,inpatient_total_util_days,inpatient_avg_diag_count,has_inpatient_claim
0,00013D2EFD8E45D1,1,4000.0,4000.0,4000.0,0.0,1100.0,0.0,0.0,1.0,10.00,1
1,00016F745862898F,4,52000.0,13000.0,26000.0,0.0,4304.0,0.0,0.0,16.0,7.75,1
2,00052705243EA128,1,14000.0,14000.0,14000.0,0.0,1024.0,0.0,0.0,1.0,8.00,1
3,0007F12A492FD25D,4,53000.0,13250.0,29000.0,0.0,3224.0,0.0,0.0,31.0,9.50,1
4,000B97BA2314E971,1,2000.0,2000.0,2000.0,0.0,1068.0,0.0,0.0,4.0,10.00,1


In [9]:
# create outpatient member-level details

outp_features = (
    outp.groupby("DESYNPUF_ID")
    .agg(
        outpatient_claim_count=("CLM_ID", "count"),
        outpatient_total_paid=("CLM_PMT_AMT", "sum"),
        outpatient_avg_paid=("CLM_PMT_AMT", "mean"),
        outpatient_max_paid=("CLM_PMT_AMT", "max"),
        outpatient_total_primary_payer_paid=("NCH_PRMRY_PYR_CLM_PD_AMT", "sum"),
        outpatient_total_deductible=("NCH_BENE_PTB_DDCTBL_AMT", "sum"),
        outpatient_total_coinsurance=("NCH_BENE_PTB_COINSRNC_AMT", "sum"),
        outpatient_total_blood_deductible=("NCH_BENE_BLOOD_DDCTBL_LBLTY_AM", "sum"),
        outpatient_avg_diag_count=("outpatient_diag_count", "mean")
    )
    .reset_index()
)

outp_features["has_outpatient_claim"] = 1

outp_features.head()

,DESYNPUF_ID,outpatient_claim_count,outpatient_total_paid,outpatient_avg_paid,outpatient_max_paid,outpatient_total_primary_payer_paid,outpatient_total_deductible,outpatient_total_coinsurance,outpatient_total_blood_deductible,outpatient_avg_diag_count,has_outpatient_claim
0,00013D2EFD8E45D1,1,50.0,50.000000,50.0,0.0,0.0,10.0,0.0,2.000000,1
1,00016F745862898F,2,60.0,30.000000,30.0,0.0,0.0,70.0,0.0,4.500000,1
2,0001FDD721E223DC,1,30.0,30.000000,30.0,0.0,0.0,50.0,0.0,4.000000,1
3,00024B3D2352D2D0,4,160.0,40.000000,80.0,0.0,0.0,80.0,0.0,1.500000,1
4,0002F28CE057345B,19,2920.0,153.684211,700.0,0.0,0.0,1070.0,0.0,2.894737,1


In [10]:
# check table shapes

print("inp_features shape:", inp_features.shape)
print("outp_features shape:", outp_features.shape)

print("Unique inpatient members:", inp_features["DESYNPUF_ID"].nunique())
print("Unique outpatient members:", outp_features["DESYNPUF_ID"].nunique())

inp_features shape: (37780, 12)
outp_features shape: (85272, 11)
Unique inpatient members: 37780
Unique outpatient members: 85272


In [11]:
# merge claim features onto beneficiary 2010

member_features = benef_2010.merge(inp_features, on="DESYNPUF_ID", how="left")
member_features = member_features.merge(outp_features, on="DESYNPUF_ID", how="left")

print("member_features shape:", member_features.shape)
member_features.head()

member_features shape: (112754, 53)


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,...,outpatient_claim_count,outpatient_total_paid,outpatient_avg_paid,outpatient_max_paid,outpatient_total_primary_payer_paid,outpatient_total_deductible,outpatient_total_coinsurance,outpatient_total_blood_deductible,outpatient_avg_diag_count,has_outpatient_claim
0,00013D2EFD8E45D1,1970-01-01 00:00:00.019230501,NaT,1,1,0,26,950,12,12,...,1.0,50.0,50.0,50.0,0.0,0.0,10.0,0.0,2.0,1.0
1,00016F745862898F,1970-01-01 00:00:00.019430101,NaT,1,1,Y,39,230,12,12,...,2.0,60.0,30.0,30.0,0.0,0.0,70.0,0.0,4.5,1.0
2,0001FDD721E223DC,1970-01-01 00:00:00.019360901,NaT,2,1,0,39,280,12,12,...,1.0,30.0,30.0,30.0,0.0,0.0,50.0,0.0,4.0,1.0
3,00021CA6FF03E670,1970-01-01 00:00:00.019410601,NaT,1,5,0,6,290,12,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00024B3D2352D2D0,1970-01-01 00:00:00.019360801,NaT,1,1,0,52,590,9,12,...,4.0,160.0,40.0,80.0,0.0,0.0,80.0,0.0,1.5,1.0


In [12]:
# Fill claim-related missing values with zeros

claim_feature_cols = [
    "inpatient_claim_count",
    "inpatient_total_paid",
    "inpatient_avg_paid",
    "inpatient_max_paid",
    "inpatient_total_primary_payer_paid",
    "inpatient_total_deductible",
    "inpatient_total_coinsurance",
    "inpatient_total_blood_deductible",
    "inpatient_total_util_days",
    "inpatient_avg_diag_count",
    "has_inpatient_claim",
    "outpatient_claim_count",
    "outpatient_total_paid",
    "outpatient_avg_paid",
    "outpatient_max_paid",
    "outpatient_total_primary_payer_paid",
    "outpatient_total_deductible",
    "outpatient_total_coinsurance",
    "outpatient_total_blood_deductible",
    "outpatient_avg_diag_count",
    "has_outpatient_claim"
]

member_features[claim_feature_cols] = member_features[claim_feature_cols].fillna(0)

In [13]:
# create age in 2010

member_features["age_2010"] = (
    (pd.Timestamp("2010-12-31") - pd.to_datetime(member_features["BENE_BIRTH_DT"], errors="coerce"))
    .dt.days / 365.25
).astype("float")

In [14]:
# Create chronic condition count

chronic_cols = [
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA"
]

for col in chronic_cols:
    member_features[col] = member_features[col].replace({1: 1, 2: 0, "1": 1, "2": 0})

member_features["chronic_condition_count"] = member_features[chronic_cols].fillna(0).sum(axis=1)

In [15]:
# Create a simple target variable

member_features["total_paid_all_claims"] = (
    member_features["inpatient_total_paid"] + member_features["outpatient_total_paid"]
)

high_cost_threshold = member_features["total_paid_all_claims"].quantile(0.75)

member_features["high_cost_member"] = (
    member_features["total_paid_all_claims"] >= high_cost_threshold
).astype(int)

print("High cost threshold:", high_cost_threshold)
print(member_features["high_cost_member"].value_counts())

High cost threshold: 8200.0
high_cost_member
0    84552
1    28202
Name: count, dtype: int64


In [16]:
# quick exploratory checks

print(member_features[[
    "age_2010",
    "chronic_condition_count",
    "inpatient_claim_count",
    "outpatient_claim_count",
    "inpatient_total_paid",
    "outpatient_total_paid",
    "total_paid_all_claims"
]].describe())

           age_2010  chronic_condition_count  inpatient_claim_count  \
count  1.127540e+05            112754.000000          112754.000000   
mean   4.099384e+01                 1.800770               0.584946   
std    1.779917e-11                 1.988318               1.106214   
min    4.099384e+01                 0.000000               0.000000   
25%    4.099384e+01                 0.000000               0.000000   
50%    4.099384e+01                 1.000000               0.000000   
75%    4.099384e+01                 3.000000               1.000000   
max    4.099384e+01                11.000000              14.000000   

       outpatient_claim_count  inpatient_total_paid  outpatient_total_paid  \
count           112754.000000         112754.000000          112754.000000   
mean                 6.929244           5600.853806            1964.241446   
std                  8.194271          13132.368467            5307.449284   
min                  0.000000          -3000.000

In [17]:
# ssave the member-level feature table

member_features.to_csv(os.path.join(processed_dir, "member_features_2010.csv"), index=False)

print("Saved:", os.path.join(processed_dir, "member_features_2010.csv"))

Saved: C:\Users\Kimi\OneDrive\Desktop\PORTFOLIO PROJECTS\Health_Insurance_claims_member_engagement\data_processed\member_features_2010.csv


In [18]:
# optional save of two claim feature tables

inp_features.to_csv(os.path.join(processed_dir, "inpatient_member_features.csv"), index=False)
outp_features.to_csv(os.path.join(processed_dir, "outpatient_member_features.csv"), index=False)